# Introduction to privJedAI

This notebook will guide you throughout all the possible methods and how to use them by our open-source library privJedAI.

In [1]:
%pip install privjedai


  Using cached bitarray-3.8.0-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (34 kB)
  Using cached faiss_cpu-1.13.2-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (7.6 kB)
  Using cached matplotlib-3.10.8-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached metafone-0.5-py3-none-any.whl
  Using cached numexpr-2.14.1-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.0 kB)
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
  Using cached openfhe-1.5.0.0.22.4-py3-none-any.whl.metadata (671 bytes)
  Using cached ordered_set-4.1.0-py3-none-any.whl.metadata (5.3 kB)
  Using cached pandas-2.3.3-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.

## Import Dataset Abt Buy Clean

Below we load the two different datasets and their ground truth. privJedAI needs the indices of the pairs for each dataset and not their id. Here we present also how to preprocess a ground truth that contains id1-id2 pairs to index1-index2 pairs.

In [1]:
import pandas as pd

DIR = "D2"
PATH = f"data/ccer/{DIR}"
FILE = 'abtclean'
FILE2 = 'buyclean'
attributes = ['name']
SEP = "|"

d1 = pd.read_csv(f"../{PATH}/{FILE}.csv", sep=SEP)
d2 = pd.read_csv(f"../{PATH}/{FILE2}.csv", sep=SEP)

gt = pd.read_csv(f'../{PATH}/gtclean.csv' , sep=SEP)
df_a = d1.reset_index().rename(columns={"index": "index_A"})
df_b = d2.reset_index().rename(columns={"index": "index_B"})

df_a = df_a[["index_A", "id"]]
df_b = df_b[["index_B", "id"]]

gt_index = gt.merge(left_on='D1', right=df_a, right_on='id')

gt_index = gt_index.drop(columns=['id', 'D1'])
gt_index.columns = ['D2', 'D1']

gt_index = gt_index.merge(left_on='D2', right=df_b, right_on='id')
gt_index = gt_index.drop(columns=['id', 'D2'])
gt_index.columns = ['D1', 'D2']
d1 = d1.astype(str)
d2 = d2.astype(str)


## Encode data and build bloom filters

Each party agree in an exact configuration and then encode locally their data.
Those encoded data are then shared to a third party to proceed with record linkage.

In [2]:
from privjedai.encoder import BloomFilterConfig, BloomEncodedData, BloomFilter

bloom_filter_configuration = {
    "size" : 512,
    "offset" : 0,
    "num_hashes" : 15,
    "hashing_type": "salted_qgrams",
    "salt" : "",
    "attributes": ['name'],
    "qgrams": 4
}

config = BloomFilterConfig(**bloom_filter_configuration)
bloom_generator = BloomFilter(config)

## The two parties encode their datasets and save them to disk.
## The encoded datasets are then shared with the other party and used for the matching process.
encoded_d1 = bloom_generator.encode(d1)
encoded_d1.to_file(f"dataset_1.pkl")

encoded_d2 = bloom_generator.encode(d2)
encoded_d2.to_file(f"dataset_2.pkl")

Encoding Data with attributes ['name']:   0%|          | 0/1076 [00:00<?, ?it/s]

Encoding Data with attributes ['name']:   0%|          | 0/1064 [00:00<?, ?it/s]

In [3]:
# Third party loads the encoded datasets and performs the matching process.
encoded_data = BloomEncodedData.from_file("dataset_1.pkl", "dataset_2.pkl")


# Ground truth must be explicitly set for the evaluation process. 
# This is done by providing the indices of the matching records in the original datasets.
encoded_data.set_ground_truth(gt_index)

## Blocking with privJedAI

In privJedAI we have 2 different implementations of Hamming Blocking and a FAISS implementation.

### BitBlocker

In [4]:
from privjedai.blocking import BitBlocker

blocker = BitBlocker(psi = 8,
            lmbda = 24,
            seed = 42)

blocks = blocker.build_blocks(encoded_data=encoded_data)

_ = blocker.evaluate(blocks)



***************************************************************************************************************************
                                         Method:  BitBlocker
***************************************************************************************************************************
Method name: BitBlocker
Parameters: 
Runtime: 0.1832 seconds
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Performance:
	Precision:      0.42% 
	Recall:        85.81%
	F1-score:       0.83%
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


### LSHBlocker


In [5]:
from privjedai.blocking import LSHBlocker

blocker = LSHBlocker(psi = 8,
            lmbda = 24,
            seed = 42,
            prune_ratio = 0.8)

blocks = blocker.build_blocks(encoded_data=encoded_data)
_ = blocker.evaluate(blocks)

***************************************************************************************************************************
                                         Method:  LSHBlocker
***************************************************************************************************************************
Method name: LSHBlocker
Parameters: 
	psi: 8
	lmbda: 24
	prune_ratio: 0.8
	prune_sample: 1000
	seed: 42
Runtime: 0.2633 seconds
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Performance:
	Precision:      0.09% 
	Recall:        92.20%
	F1-score:       0.19%
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


### FAISSBlocking

In [24]:
from privjedai.blocking import FAISSBlocking

blocker = FAISSBlocking(index_type='hnsw')

blocks = blocker.build_blocks(encoded_data=encoded_data, top_k=20)
_ = blocker.evaluate(blocks)

***************************************************************************************************************************
                                         Method:  FAISS Blocking
***************************************************************************************************************************
Method name: FAISS Blocking
Parameters: 
Runtime: 0.0714 seconds
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Performance:
	Precision:      4.40% 
	Recall:        88.06%
	F1-score:       8.39%
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


## Meta-blocking Techniques

Those above are all standard blocking techniques. privJedAI also implements meta-blocking methods. It leverages comparison cleaning methods from ER and implements them for bitarrays. A block is a set of adjacent active bits of a bitarray.

In [27]:
from privjedai.comparison_cleaning import CardinalityEdgePruning

cc = CardinalityEdgePruning(weighting_scheme='CN-CBS')

cc_blocks = cc.process(encoded_data, adjacent_bits=2)

_ = cc.evaluate(cc_blocks)

Total matching pairs: 233036
***************************************************************************************************************************
                                         Method:  Cardinality Edge Pruning
***************************************************************************************************************************
Method name: Cardinality Edge Pruning
Parameters: 
	Node centric: False
	Weighting scheme: CN-CBS
Runtime: 0.7187 seconds
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Performance:
	Precision:      0.45% 
	Recall:        98.50%
	F1-score:       0.90%
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


## Matching

After filtering our datasets we can use a similarity function and match the possible candidate pairs.

In [25]:
from privjedai.matching import Matcher
import numpy as np
matcher = Matcher(batch_size = 10_000,
                  threshold = 0.6,
                  metric='cosine')

matches = matcher.predict(encoded_data=encoded_data, blocks=blocks)

_ = matcher.evaluate(matches)

Predicting batches:   0%|          | 0/3 [00:00<?, ?it/s]

***************************************************************************************************************************
                                         Method:  Matcher
***************************************************************************************************************************
Method name: Matcher
Parameters: 
	batch_size: 10000
	threshold: 0.6
	metric: cosine
	attributes: None
Runtime: 0.0830 seconds
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Performance:
	Precision:      4.76% 
	Recall:        87.59%
	F1-score:       9.03%
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


## Clustering

To eliminate possible conflicts on the matching pairs, we provide multiple clustering techniques.

In [26]:
from privjedai.clustering import KiralyMSMApproximateClustering

clusterer = KiralyMSMApproximateClustering()

clusters = clusterer.process(matches, encoded_data=encoded_data, similarity_threshold=0.5)

_ = clusterer.evaluate(clusters)

***************************************************************************************************************************
                                         Method:  Kiraly MSM Approximate Clustering
***************************************************************************************************************************
Method name: Kiraly MSM Approximate Clustering
Parameters: 
	Similarity Threshold: 0.5
Runtime: 0.0273 seconds
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Performance:
	Precision:     79.38% 
	Recall:        72.74%
	F1-score:      75.92%
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
